# Graph-Augmented Intelligence
## Notebook 02: GDS Algorithms via Python Driver

Run Neo4j Graph Data Science algorithms directly from Databricks using the
`graphdatascience` Python client — no manual Cypher in the Aura UI needed.

After running `01_neo4j_ingest`, execute this notebook to compute three
fraud-signal features on every Account node:

| Algorithm | Property Written | Fraud Signal |
|-----------|-----------------|-------------|
| **PageRank** | `Account.risk_score` | Central accounts in money-flow networks |
| **Louvain** | `Account.community_id` | Tightly connected clusters (fraud rings) |
| **Node Similarity** | `Account.similarity_score` | Accounts sharing the same merchants |

> **Prerequisite:** AuraDB Professional with the GDS Plugin enabled.

In [ ]:
%pip install graphdatascience --quiet

In [ ]:
dbutils.library.restartPython()

## 0. Configuration and Connect

Run after the `%pip install` kernel restart above.

In [ ]:
CATALOG   = "graph-enriched-lakehouse"
SCHEMA    = "graph-enriched-schema"

NEO4J_URI      = dbutils.secrets.get("neo4j-graph-engineering", "uri")
NEO4J_USER     = dbutils.secrets.get("neo4j-graph-engineering", "username")
NEO4J_PASSWORD = dbutils.secrets.get("neo4j-graph-engineering", "password")

In [ ]:
from graphdatascience import GraphDataScience

gds = GraphDataScience(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
print(f"Connected  |  GDS version: {gds.version()}")

---
## Step 1: Verify the Graph Loaded Correctly

In [ ]:
counts = gds.run_cypher("""
    MATCH (a:Account) WITH count(a) AS accounts
    MATCH (m:Merchant) WITH accounts, count(m) AS merchants
    MATCH ()-[t:TRANSACTED_WITH]->() WITH accounts, merchants, count(t) AS txns
    MATCH ()-[p:TRANSFERRED_TO]->() WITH accounts, merchants, txns, count(p) AS p2p
    RETURN accounts, merchants, txns, p2p
""")
print(counts.to_string(index=False))

---
## Step 2: Project the Account Transfer Graph

GDS algorithms run on an in-memory projection. This projects only Account
nodes and TRANSFERRED_TO relationships — the peer-to-peer money-flow graph
where fraud rings operate.

The projection is UNDIRECTED because money-mule rings form cycles; directionality
adds noise for community detection and PageRank on this dataset.

In [ ]:
# Idempotent: safe to re-run — drops stale projection if it exists
gds.run_cypher("CALL gds.graph.drop('account_transfers', false) YIELD graphName")

G, stats = gds.graph.project(
    "account_transfers",
    "Account",
    {"TRANSFERRED_TO": {"orientation": "UNDIRECTED"}},
)
print(f"Projected '{G.name()}': {stats['nodeCount']:,} nodes, {stats['relationshipCount']:,} relationships")

---
## Step 3: PageRank — Risk Centrality

PageRank measures how central an account is in the transfer network.
Accounts that receive transfers from other high-PageRank accounts score higher —
the structural signature of money-mule networks.

Fraud rings score high because they receive from **other ring members**, not from
peripheral accounts. High-volume "whale" hubs score low because their senders
are unimportant accounts.

In [ ]:
pr = gds.pageRank.write(
    G,
    maxIterations=20,
    dampingFactor=0.85,
    writeProperty="risk_score",
)
print(f"Properties written: {int(pr['nodePropertiesWritten']):,}")
print(f"Iterations:         {int(pr['ranIterations'])}")
print(f"Converged:          {bool(pr['didConverge'])}")

top_pr = gds.run_cypher("""
    MATCH (a:Account)
    WHERE a.risk_score IS NOT NULL
    RETURN a.account_id AS id, round(a.risk_score, 6) AS pagerank
    ORDER BY a.risk_score DESC LIMIT 10
""")
print("\nTop 10 accounts by PageRank:")
print(top_pr.to_string(index=False))

---
## Step 4: Louvain Community Detection — Fraud Rings

Louvain finds clusters of densely connected accounts. Fraud rings form small,
tight communities with heavy internal transfers. Legitimate accounts form large,
diffuse communities.

In [ ]:
louvain = gds.louvain.write(
    G,
    writeProperty="community_id",
)
print(f"Communities found:  {int(louvain['communityCount']):,}")
print(f"Modularity:         {float(louvain['modularity']):.4f}")
print(f"Properties written: {int(louvain['nodePropertiesWritten']):,}")

community_sizes = gds.run_cypher("""
    MATCH (a:Account)
    WHERE a.community_id IS NOT NULL
    RETURN a.community_id AS community, count(*) AS size
    ORDER BY size DESC LIMIT 15
""")
print("\nTop 15 communities by size (small, dense ones are fraud rings):")
print(community_sizes.to_string(index=False))

---
## Step 5: Drop the Transfer Graph Projection

In [ ]:
gds.graph.drop(G)
print(f"Dropped: {G.name()}")

---
## Step 6: Project the Bipartite Account–Merchant Graph

Node Similarity needs the bipartite graph: which accounts transact with which merchants.
The NATURAL orientation preserves the Account → Merchant direction.

In [ ]:
# Idempotent: safe to re-run
gds.run_cypher("CALL gds.graph.drop('account_merchants', false) YIELD graphName")

G2, stats2 = gds.graph.project(
    "account_merchants",
    ["Account", "Merchant"],
    {"TRANSACTED_WITH": {"orientation": "NATURAL"}},
)
print(f"Projected '{G2.name()}': {stats2['nodeCount']:,} nodes, {stats2['relationshipCount']:,} relationships")

---
## Step 7: Node Similarity — Shared Merchant Patterns

Two accounts are similar if they transact at the same merchants (Jaccard similarity
on their merchant neighborhoods). Fraud accounts within a ring share a set of
anchor merchants, producing high intra-ring similarity scores.

In [ ]:
ns = gds.nodeSimilarity.write(
    G2,
    similarityMetric="JACCARD",
    topK=10,
    writeRelationshipType="SIMILAR_TO",
    writeProperty="similarity_score",
)
print(f"Nodes compared:        {int(ns['nodesCompared']):,}")
print(f"Relationships written: {int(ns['relationshipsWritten']):,}")

top_pairs = gds.run_cypher("""
    MATCH (a:Account)-[s:SIMILAR_TO]-(b:Account)
    WHERE a.account_id < b.account_id
    RETURN a.account_id AS account_a, b.account_id AS account_b,
           round(s.similarity_score, 3) AS similarity
    ORDER BY s.similarity_score DESC LIMIT 10
""")
print("\nTop 10 similar account pairs by Jaccard similarity:")
print(top_pairs.to_string(index=False))

---
## Step 8: Aggregate Max Similarity per Account

Store each account's highest similarity score as a node property. This collapses
the per-relationship scores to a single value that reads back as one column
when pulling features into Databricks.

In [ ]:
agg = gds.run_cypher("""
    MATCH (a:Account)-[s:SIMILAR_TO]-()
    WITH a, MAX(s.similarity_score) AS max_sim
    SET a.similarity_score = max_sim
    RETURN count(a) AS accounts_updated
""")
print(f"Accounts updated: {int(agg['accounts_updated'].iloc[0]):,}")

---
## Step 9: Drop the Bipartite Graph Projection

In [ ]:
gds.graph.drop(G2)
print(f"Dropped: {G2.name()}")

---
## Step 10: Final Verification — All Features Written

Confirm all three properties are set on every Account node and check for
fraud vs legitimate separation. You should see clear separation in `avg_pagerank`
and `avg_similarity` — that separation is the ML lift we measure in notebook 03.

In [ ]:
complete = gds.run_cypher("""
    MATCH (a:Account)
    WHERE a.risk_score IS NOT NULL
      AND a.community_id IS NOT NULL
      AND a.similarity_score IS NOT NULL
    RETURN count(a) AS accounts_with_all_features
""")
print(f"Accounts with all three features: {int(complete['accounts_with_all_features'].iloc[0]):,}")

feature_summary = gds.run_cypher("""
    MATCH (a:Account)
    RETURN count(a) AS accounts,
           round(avg(a.risk_score), 6) AS avg_pagerank,
           round(avg(a.similarity_score), 4) AS avg_similarity,
           count(DISTINCT a.community_id) AS num_communities
""")
print("\nFeature summary across all accounts:")
print(feature_summary.to_string(index=False))

---
## Done

All three GDS features are now written on every Account node:

| Property | Algorithm | Expected signal |
|----------|-----------|----------------|
| `risk_score` | PageRank | Higher for fraud ring members than whale hubs |
| `community_id` | Louvain | Ring members share a small, dense community |
| `similarity_score` | Node Similarity | High between accounts sharing anchor merchants |

**Next →** Run `03_pull_gold_tables` to read these features back into Databricks,
train a baseline and a graph-augmented model, and measure the ML lift.